# Feature Selection - NHANES Diabetes Prediction

Comparison of **5 feature selection methods** to identify the most important predictors.

## Implemented Methods:
1. **Mutual Information** - Non-linear relationships
2. **Lasso** - L1 regularization
3. **RFE** - Recursive Feature Elimination
4. **Random Forest** - Gini-based importance
5. **XGBoost** - Feature gain in gradient boosting
6. **Ensemble** - Democratic voting across methods

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All imports successful")

## Load Processed Data

In [ ]:
# Load data (assumes it's already clean, imputed, encoded)
df = pd.read_parquet("../data/dataset/dataset_prepared.parquet")


print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['diabetes_risk'].value_counts())
print(f"\nFeatures: {df.columns.tolist()}")

## Prepare X, y

In [ ]:
# Separate features and target
X = df.drop('diabetes_risk', axis=1)
y = df['diabetes_risk']

# Convert target to numeric if necessary
if y.dtype == 'object':
    y = (y == 'Yes').astype(int)  # Adjust according to your encoding

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX dtypes:")
print(X.dtypes.value_counts())
print(f"\nTarget classes: {y.nunique()}")

## MUTUAL INFORMATION

In [ ]:
print("="*80)
print("MUTUAL INFORMATION")
print("="*80)

# Calculate MI scores
mi_scores = mutual_info_classif(X, y, random_state=42)

# DataFrame with results
mi_df = pd.DataFrame({
    'feature': X.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

print(f"\nTop 20 features by Mutual Information:")
print(mi_df.head(20).to_string(index=False))

mi_top_20 = mi_df['feature'].head(20).tolist()

### MI Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
mi_top = mi_df.head(20)
ax.barh(range(len(mi_top)), mi_top['mi_score'].values, color='steelblue')
ax.set_yticks(range(len(mi_top)))
ax.set_yticklabels(mi_top['feature'].values)
ax.set_xlabel('Mutual Information Score', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - Mutual Information', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(mi_top['mi_score'].values):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

print(f"✓ Top 20 MI features: {len(mi_top_20)} features")

## LASSO (L1 REGULARIZATION)

In [ ]:
print("="*80)
print("LASSO (L1 REGULARIZATION)")
print("="*80)

# Scale features (required for Lasso)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Fit Logistic Regression with L1 penalty
print("\nFitting Logistic Regression with L1 penalty...")
lasso = LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=2000, random_state=42, n_jobs=-1)
lasso.fit(X_scaled_df, y)

# Get coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': np.abs(lasso.coef_[0])
}).sort_values('coefficient', ascending=False)

# Non-zero features
non_zero = coef_df[coef_df['coefficient'] > 0]

print(f"\nNon-zero features: {len(non_zero)} / {len(coef_df)}")
print(f"\nTop 20 features by Lasso coefficient:")
print(coef_df.head(20).to_string(index=False))

lasso_top_20 = coef_df['feature'].head(20).tolist()

### Lasso Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
lasso_top = coef_df.head(20)
colors = ['darkred' if x == 0 else 'forestgreen' for x in lasso_top['coefficient'].values]
ax.barh(range(len(lasso_top)), lasso_top['coefficient'].values, color=colors)
ax.set_yticks(range(len(lasso_top)))
ax.set_yticklabels(lasso_top['feature'].values)
ax.set_xlabel('|Coefficient|', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - Lasso (L1)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(lasso_top['coefficient'].values):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()

print(f"✓ Top 20 Lasso features: {len(lasso_top_20)} features")

## RFE (RECURSIVE FEATURE ELIMINATION)

In [ ]:
print("="*80)
print("RFE (RECURSIVE FEATURE ELIMINATION)")
print("="*80)

# Base estimator
base_est = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# RFE to select top 20
print("\nFitting RFE with Random Forest estimator...")
rfe = RFE(estimator=base_est, n_features_to_select=20, step=5)
rfe.fit(X, y)

# Get results
rfe_df = pd.DataFrame({
    'feature': X.columns,
    'ranking': rfe.ranking_,
    'selected': rfe.support_
}).sort_values('ranking')

selected_features = rfe_df[rfe_df['selected']]['feature'].tolist()

print(f"\nSelected {len(selected_features)} features:")
print(rfe_df.head(20).to_string(index=False))

rfe_top_20 = selected_features

### RFE Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
rfe_top = rfe_df.head(20)
colors = ['forestgreen' if x else 'lightcoral' for x in rfe_top['selected'].values]
ax.barh(range(len(rfe_top)), -rfe_top['ranking'].values, color=colors)
ax.set_yticks(range(len(rfe_top)))
ax.set_yticklabels(rfe_top['feature'].values)
ax.set_xlabel('Ranking (lower = better)', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - RFE', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(rfe_top['ranking'].values):
    ax.text(-v - 0.3, i, f'{v}', va='center')
plt.tight_layout()
plt.show()

print(f"✓ Top 20 RFE features: {len(rfe_top_20)} features")

## RANDOM FOREST IMPORTANCE

In [ ]:
print("="*80)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("="*80)

# Train Random Forest
print("\nTraining Random Forest (200 trees)...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get importance
rf_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_,
    'importance_pct': (rf.feature_importances_ / rf.feature_importances_.sum()) * 100
}).sort_values('importance', ascending=False)

print(f"\nTop 20 features by Random Forest importance:")
print(rf_df.head(20).to_string(index=False))

# Cumsum
rf_df['cumsum_pct'] = rf_df['importance_pct'].cumsum()
print(f"\nCumulative importance (top 20): {rf_df.head(20)['cumsum_pct'].iloc[-1]:.2f}%")

rf_top_20 = rf_df['feature'].head(20).tolist()

### Random Forest Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
rf_top = rf_df.head(20)
ax.barh(range(len(rf_top)), rf_top['importance'].values, color='forestgreen')
ax.set_yticks(range(len(rf_top)))
ax.set_yticklabels(rf_top['feature'].values)
ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - Random Forest', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, (v, pct) in enumerate(zip(rf_top['importance'].values, rf_top['importance_pct'].values)):
    ax.text(v + 0.0003, i, f'{pct:.2f}%', va='center')
plt.tight_layout()
plt.show()

print(f"✓ Top 20 RF features: {len(rf_top_20)} features")

## XGBOOST GAIN

In [ ]:
print("="*80)
print("XGBOOST FEATURE IMPORTANCE (GAIN)")
print("="*80)

# Train XGBoost
print("\nTraining XGBoost (200 rounds)...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
xgb_model.fit(X, y)

# Get importance (gain)
importance_dict = xgb_model.get_booster().get_score(importance_type='gain')

xgb_df = pd.DataFrame({
    'feature': list(importance_dict.keys()),
    'gain': list(importance_dict.values())
}).sort_values('gain', ascending=False)

xgb_df['gain_pct'] = (xgb_df['gain'] / xgb_df['gain'].sum()) * 100

print(f"\nTop 20 features by XGBoost Gain:")
print(xgb_df.head(20).to_string(index=False))

# Cumsum
xgb_df['cumsum_pct'] = xgb_df['gain_pct'].cumsum()
print(f"\nCumulative gain (top 20): {xgb_df.head(20)['cumsum_pct'].iloc[-1]:.2f}%")

xgb_top_20 = xgb_df['feature'].head(20).tolist()

### XGBoost Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
xgb_top = xgb_df.head(20)
ax.barh(range(len(xgb_top)), xgb_top['gain'].values, color='darkblue')
ax.set_yticks(range(len(xgb_top)))
ax.set_yticklabels(xgb_top['feature'].values)
ax.set_xlabel('Gain', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - XGBoost', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for i, (v, pct) in enumerate(zip(xgb_top['gain'].values, xgb_top['gain_pct'].values)):
    ax.text(v + 5, i, f'{pct:.2f}%', va='center')
plt.tight_layout()
plt.show()

print(f"✓ Top 20 XGB features: {len(xgb_top_20)} features")

## ENSEMBLE VOTING

### Voting System

In [ ]:
print("="*80)
print("ENSEMBLE FEATURE SELECTION (VOTING)")
print("="*80)

# Collect top 20 from each method
method_votes = {
    'MI': mi_top_20,
    'Lasso': lasso_top_20,
    'RFE': rfe_top_20,
    'RF': rf_top_20,
    'XGB': xgb_top_20
}

# Count votes
votes = {}
for method_name, features in method_votes.items():
    for rank, feature in enumerate(features, 1):
        if feature not in votes:
            votes[feature] = {'count': 0, 'ranks': []}
        votes[feature]['count'] += 1
        votes[feature]['ranks'].append((method_name, rank))

# Create ensemble ranking
ensemble_df = pd.DataFrame([
    {
        'feature': feature,
        'vote_count': data['count'],
        'avg_rank': np.mean([r[1] for r in data['ranks']]),
        'methods': ', '.join([r[0] for r in data['ranks']])
    }
    for feature, data in votes.items()
]).sort_values(['vote_count', 'avg_rank'], ascending=[False, True])

# Consensus features (voted by 3+ methods)
consensus_features = ensemble_df[ensemble_df['vote_count'] >= 3]['feature'].tolist()

print(f"\nTotal unique features in top 20 lists: {len(votes)}")
print(f"Consensus features (3+ votes): {len(consensus_features)}")
print(f"\nTop 20 by consensus:")
print(ensemble_df.head(20).to_string(index=False))

### Ensemble Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ens_top = ensemble_df.head(20)

# Color by vote count
colors = ['darkgreen' if x >= 4 else 'forestgreen' if x >= 3 else 'orange' for x in ens_top['vote_count'].values]

ax.barh(range(len(ens_top)), ens_top['vote_count'].values, color=colors)
ax.set_yticks(range(len(ens_top)))
ax.set_yticklabels(ens_top['feature'].values)
ax.set_xlabel('Vote Count (max: 5)', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features - Ensemble Voting', fontsize=14, fontweight='bold')
ax.set_xlim(0, 5.5)
ax.invert_yaxis()

for i, (v, methods) in enumerate(zip(ens_top['vote_count'].values, ens_top['methods'].values)):
    ax.text(v + 0.1, i, f'{int(v)}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✓ Ensemble top 20 features selected")

### Comparison Table

In [ ]:
# Heatmap of votes
top_features = ensemble_df['feature'].head(20).tolist()

# Create voting matrix
vote_matrix = []
for feature in top_features:
    row = []
    for method, features_list in method_votes.items():
        if feature in features_list:
            rank = features_list.index(feature) + 1
            row.append(rank)
        else:
            row.append(np.nan)
    vote_matrix.append(row)

vote_heatmap = pd.DataFrame(
    vote_matrix,
    index=top_features,
    columns=['MI', 'Lasso', 'RFE', 'RF', 'XGB']
)

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(vote_heatmap, annot=True, fmt='.0f', cmap='RdYlGn_r', cbar_kws={'label': 'Rank (lower=better)'}, ax=ax)
ax.set_title('Feature Ranking by Method (Top 20 Ensemble)', fontsize=14, fontweight='bold')
ax.set_xlabel('Methods', fontsize=12, fontweight='bold')
ax.set_ylabel('Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## SUMMARY & RECOMMENDATIONS

In [ ]:
print("\n" + "="*80)
print("FEATURE SELECTION SUMMARY")
print("="*80)

print(f"\n📊 STATISTICS:")
print(f"  • Total features in dataset: {X.shape[1]}")
print(f"  • Features in top 20 (any method): {len(votes)}")
print(f"  • Consensus features (3+ votes): {len(consensus_features)}")
print(f"  • Consensus features (4+ votes): {len(ensemble_df[ensemble_df['vote_count'] >= 4])}")
print(f"  • Unanimous features (5 votes): {len(ensemble_df[ensemble_df['vote_count'] == 5])}")

print(f"\n🏆 TOP CONSENSUS FEATURES (5 votes - all methods agree):")
unanimous = ensemble_df[ensemble_df['vote_count'] == 5]
for idx, row in unanimous.iterrows():
    print(f"  ✓ {row['feature']:30s} (avg_rank: {row['avg_rank']:.1f})")

print(f"\n🎯 RECOMMENDED FEATURE SET:")
recommended = ensemble_df[ensemble_df['vote_count'] >= 3].head(20)
print(f"  Features: {len(recommended)} features with 3+ consensus votes")
print(recommended[['feature', 'vote_count', 'avg_rank']].to_string(index=False))

print(f"\n💾 FEATURES TO USE FOR MODELING:")
final_features = recommended['feature'].tolist()
print(f"final_features = {final_features}")

## Create Final Dataset

In [ ]:
# Dataset with selected features
final_features = ensemble_df[ensemble_df['vote_count'] >= 3]['feature'].head(20).tolist()

df_selected = df[final_features + ['diabetes_risk']].copy()

print(f"Original shape: {df.shape}")
print(f"Selected shape: {df_selected.shape}")
print(f"\nFeatures selected: {len(final_features)}")
print(f"Dimensionality reduction: {((df.shape[1] - df_selected.shape[1]) / df.shape[1] * 100):.1f}%")
"""
# Save
df_selected.to_parquet("./data/nhanes_data/NHANES_selected_features.parquet", index=False)
print(f"\n✓ Saved to NHANES_selected_features.parquet")

# Also save feature list
import json
with open("./data/selected_features.json", 'w') as f:
    json.dump({'selected_features': final_features}, f, indent=2)
print(f"✓ Saved feature list to selected_features.json")
"""

## Final Visualization

In [ ]:
# Method comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# MI
axes[0, 0].barh(range(10), mi_df.head(10)['mi_score'].values, color='steelblue')
axes[0, 0].set_yticks(range(10))
axes[0, 0].set_yticklabels(mi_df.head(10)['feature'].values)
axes[0, 0].set_title('Mutual Information', fontweight='bold')
axes[0, 0].invert_yaxis()

# Lasso
axes[0, 1].barh(range(10), coef_df.head(10)['coefficient'].values, color='forestgreen')
axes[0, 1].set_yticks(range(10))
axes[0, 1].set_yticklabels(coef_df.head(10)['feature'].values)
axes[0, 1].set_title('Lasso Coefficients', fontweight='bold')
axes[0, 1].invert_yaxis()

# RFE
rfe_top_10 = rfe_df.head(10)
axes[0, 2].barh(range(10), -rfe_top_10['ranking'].values, color='orange')
axes[0, 2].set_yticks(range(10))
axes[0, 2].set_yticklabels(rfe_top_10['feature'].values)
axes[0, 2].set_title('RFE Ranking', fontweight='bold')
axes[0, 2].invert_yaxis()

# RF
axes[1, 0].barh(range(10), rf_df.head(10)['importance'].values, color='darkred')
axes[1, 0].set_yticks(range(10))
axes[1, 0].set_yticklabels(rf_df.head(10)['feature'].values)
axes[1, 0].set_title('Random Forest', fontweight='bold')
axes[1, 0].invert_yaxis()

# XGB
axes[1, 1].barh(range(10), xgb_df.head(10)['gain'].values, color='darkblue')
axes[1, 1].set_yticks(range(10))
axes[1, 1].set_yticklabels(xgb_df.head(10)['feature'].values)
axes[1, 1].set_title('XGBoost Gain', fontweight='bold')
axes[1, 1].invert_yaxis()

# Ensemble
axes[1, 2].barh(range(10), ensemble_df.head(10)['vote_count'].values, color='purple')
axes[1, 2].set_yticks(range(10))
axes[1, 2].set_yticklabels(ensemble_df.head(10)['feature'].values)
axes[1, 2].set_xlim(0, 5.5)
axes[1, 2].set_title('Ensemble Votes', fontweight='bold')
axes[1, 2].invert_yaxis()

fig.suptitle('Feature Selection - All Methods Comparison (Top 10)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ Comparison complete")